# Experiment 11: Generic Convection-Diffusion Transport

**Aim:** Compare scalar transport in forced flat-plate, forced channel, and natural-convection surrogate velocity fields.

**Numerical method:** Finite-volume-style transient iteration of a two-dimensional convection-diffusion equation with prescribed velocity fields.

> **Scope:** The natural case uses a prescribed recirculating velocity field. It is not a fully coupled Boussinesq solution.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 11
ROOT = Path("/content/cfd_lab_exp11") if IN_COLAB else Path.cwd() / "_colab_runtime_exp11"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp11_convection_diffusion.py').write_text('from __future__ import annotations\n\nimport csv\nimport math\nimport sys\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom common import laplacian, require_finite, upwind_derivative_x, upwind_derivative_y, write_multiline_vtk, write_structured_grid_vtk\n\n\n@dataclass\nclass CaseData:\n    name: str\n    x: np.ndarray\n    y: np.ndarray\n    temperature: np.ndarray\n    u: np.ndarray\n    v: np.ndarray\n    marker: np.ndarray\n    pe: float\n    alpha: float\n    dx: float\n    dy: float\n\n\ndef initialize_case(mode: str, nx: int, ny: int, pe: float) -> CaseData:\n    if mode == "forced_plate":\n        length, height = 5.0, 1.0\n    elif mode == "forced_pipe":\n        length, height = 5.0, 1.0\n    elif mode == "natural_plate":\n        length, height = 1.0, 1.0\n    else:\n        raise ValueError("unknown case")\n    x1 = np.linspace(0.0, length, nx)\n    y1 = np.linspace(0.0, height, ny)\n    x, y = np.meshgrid(x1, y1)\n    dx = x1[1] - x1[0]\n    dy = y1[1] - y1[0]\n    temperature = np.zeros((ny, nx), dtype=float)\n    marker = np.zeros_like(temperature)\n    if mode == "forced_plate":\n        delta = 0.035 + 0.22 * np.sqrt(np.maximum(x, dx) / length)\n        u = np.tanh(y / np.maximum(delta, 1e-12))\n        v = np.zeros_like(u)\n        marker[0, x1 >= 0.25 * length] = 1.0\n        name = "forced_flat_plate"\n    elif mode == "forced_pipe":\n        eta = y / height\n        u = 6.0 * eta * (1.0 - eta)\n        v = np.zeros_like(u)\n        marker[0, :] = 1.0\n        marker[-1, :] = 1.0\n        name = "forced_pipe_channel"\n    else:\n        amplitude = 1.0 / math.pi\n        u = -amplitude * math.pi / height * np.sin(math.pi * x / length) * np.cos(math.pi * y / height)\n        v = amplitude * math.pi / length * np.cos(math.pi * x / length) * np.sin(math.pi * y / height)\n        temperature = 1.0 - x / length\n        marker[:, 0] = 1.0\n        marker[:, -1] = 2.0\n        name = "natural_vertical_plate_surrogate"\n    return CaseData(name, x, y, temperature, u, v, marker, pe, 1.0 / pe, dx, dy)\n\n\ndef apply_bc(case: CaseData) -> None:\n    t = case.temperature\n    ny, nx = t.shape\n    if case.name == "forced_flat_plate":\n        t[:, 0] = 0.0\n        t[-1, :] = 0.0\n        heated = case.x[0, :] >= 0.25 * case.x[0, -1]\n        t[0, :] = np.where(heated, 1.0, 0.0)\n        t[1:-1, -1] = t[1:-1, -2]\n    elif case.name == "forced_pipe_channel":\n        t[:, 0] = 0.0\n        t[0, :] = 1.0\n        t[-1, :] = 1.0\n        t[1:-1, -1] = t[1:-1, -2]\n    else:\n        t[:, 0] = 1.0\n        t[:, -1] = 0.0\n        t[0, 1:-1] = t[1, 1:-1]\n        t[-1, 1:-1] = t[-2, 1:-1]\n\n\ndef outlet_or_mean(case: CaseData) -> float:\n    if case.name == "natural_vertical_plate_surrogate":\n        return float(np.mean(case.temperature[1:-1, 1:-1]))\n    weights = np.maximum(case.u[1:-1, -1], 0.0)\n    if float(np.sum(weights)) < 1e-14:\n        return float(np.mean(case.temperature[1:-1, -1]))\n    return float(np.average(case.temperature[1:-1, -1], weights=weights))\n\n\ndef average_wall_flux(case: CaseData) -> float:\n    t = case.temperature\n    if case.name == "forced_flat_plate":\n        heated = case.x[0, 1:-1] >= 0.25 * case.x[0, -1]\n        flux = -(t[1, 1:-1] - t[0, 1:-1]) / case.dy\n        return float(np.mean(flux[heated]))\n    if case.name == "forced_pipe_channel":\n        bottom = np.abs((t[0, 1:-1] - t[1, 1:-1]) / case.dy)\n        top = np.abs((t[-1, 1:-1] - t[-2, 1:-1]) / case.dy)\n        return float(np.mean(np.concatenate((bottom, top))))\n    flux = -(t[1:-1, 1] - t[1:-1, 0]) / case.dx\n    return float(np.mean(flux))\n\n\ndef run_case(case: CaseData, iterations: int, output_every: int) -> list[object]:\n    apply_bc(case)\n    max_u = max(float(np.max(np.abs(case.u))), 1e-12)\n    max_v = max(float(np.max(np.abs(case.v))), 1e-12)\n    dt_convective = 0.35 * min(case.dx / max_u, case.dy / max_v)\n    dt_diffusive = 0.20 * min(case.dx * case.dx, case.dy * case.dy) / case.alpha\n    dt = min(dt_convective, dt_diffusive)\n    history: list[list[float]] = []\n    residual = float("inf")\n    for iteration in range(1, iterations + 1):\n        old = case.temperature.copy()\n        convection = case.u * upwind_derivative_x(old, case.u, case.dx) + case.v * upwind_derivative_y(old, case.v, case.dy)\n        updated = old + dt * (-convection + case.alpha * laplacian(old, case.dx, case.dy))\n        case.temperature[1:-1, 1:-1] = np.clip(updated[1:-1, 1:-1], -0.05, 1.05)\n        apply_bc(case)\n        residual = float(np.sqrt(np.mean((case.temperature[1:-1, 1:-1] - old[1:-1, 1:-1]) ** 2)))\n        if iteration == 1 or iteration == iterations or iteration % output_every == 0:\n            outlet = outlet_or_mean(case)\n            flux = average_wall_flux(case)\n            history.append([iteration, residual, outlet, flux, dt])\n            print(f"case={case.name} iteration={iteration} residual={residual:.6e} outlet_or_mean_temperature={outlet:.6e} wall_flux={flux:.6e}")\n        require_finite(case.temperature)\n        if residual < 1e-9 and iteration > 100:\n            break\n    speed = np.sqrt(case.u * case.u + case.v * case.v)\n    local_pe = speed / max(case.alpha, 1e-30)\n    write_structured_grid_vtk(f"{case.name}.vtk", case.x, case.y, {"temperature": case.temperature, "speed": speed, "wall_or_boundary_marker": case.marker, "local_peclet_speed_based": local_pe}, {"velocity": (case.u, case.v)}, case.name)\n    length = case.x[0, -1]\n    height = case.y[-1, 0]\n    if case.name in {"forced_flat_plate", "forced_pipe_channel"}:\n        lines = [(np.array([0.0, length]), np.array([0.0, 0.0])), (np.array([0.0, length]), np.array([height, height]))]\n    else:\n        lines = [(np.array([0.0, 0.0]), np.array([0.0, height])), (np.array([length, length]), np.array([0.0, height]))]\n    write_multiline_vtk(f"{case.name}_boundaries.vtk", lines, f"{case.name} boundaries")\n    with open(f"{case.name}_history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["iteration", "residual_l2", "outlet_or_mean_temperature", "average_wall_flux", "dt"])\n        writer.writerows(history)\n    i_mid = case.temperature.shape[1] // 2\n    j_mid = case.temperature.shape[0] // 2\n    with open(f"{case.name}_profiles.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["x", "y", "temperature", "u", "v", "speed"])\n        for j in range(case.temperature.shape[0]):\n            writer.writerow([case.x[j, i_mid], case.y[j, i_mid], case.temperature[j, i_mid], case.u[j, i_mid], case.v[j, i_mid], speed[j, i_mid]])\n        for i in range(case.temperature.shape[1]):\n            writer.writerow([case.x[j_mid, i], case.y[j_mid, i], case.temperature[j_mid, i], case.u[j_mid, i], case.v[j_mid, i], speed[j_mid, i]])\n    return [case.name, case.temperature.shape[1], case.temperature.shape[0], case.pe, dt, residual, outlet_or_mean(case), average_wall_flux(case)]\n\n\ndef main() -> int:\n    mode = sys.argv[1] if len(sys.argv) > 1 else "all"\n    nx = int(sys.argv[2]) if len(sys.argv) > 2 else 121\n    ny = int(sys.argv[3]) if len(sys.argv) > 3 else 61\n    pe = float(sys.argv[4]) if len(sys.argv) > 4 else 200.0\n    iterations = int(sys.argv[5]) if len(sys.argv) > 5 else 3000\n    output_every = int(sys.argv[6]) if len(sys.argv) > 6 else 300\n    if nx < 21 or ny < 21 or pe < 1.0 or iterations < 1 or output_every < 1:\n        raise ValueError("usage: python exp11_convection_diffusion.py mode nx ny Pe iterations outputEvery")\n    aliases = {\n        "forced_plate": "forced_plate",\n        "flat": "forced_plate",\n        "forced_flat_plate": "forced_plate",\n        "forced_pipe": "forced_pipe",\n        "pipe": "forced_pipe",\n        "forced_pipe_channel": "forced_pipe",\n        "natural_plate": "natural_plate",\n        "natural": "natural_plate",\n        "natural_vertical_plate_surrogate": "natural_plate",\n    }\n    modes = ["forced_plate", "forced_pipe", "natural_plate"] if mode == "all" else [aliases.get(mode, "")]\n    if not all(modes):\n        raise ValueError("mode must be all, forced_plate, forced_pipe, or natural_plate")\n    rows = []\n    for selected in modes:\n        rows.append(run_case(initialize_case(selected, nx, ny, pe), iterations, output_every))\n    with open("convective_transport_summary.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["case", "nx", "ny", "Pe", "dt", "final_residual", "outlet_or_mean_temperature", "average_wall_flux"])\n        writer.writerows(rows)\n    with open("convective_transport_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Generic 2D convection-diffusion transport equation\\n")\n        handle.write("forced_flat_plate uses prescribed external flow over a partially heated plate.\\n")\n        handle.write("forced_pipe_channel uses prescribed parabolic internal flow between heated walls.\\n")\n        handle.write("natural_vertical_plate_surrogate uses a prescribed divergence-free recirculating field.\\n")\n        handle.write("The natural-convection case is not a coupled Navier-Stokes and energy solution.\\n")\n    print("Wrote convection-diffusion VTK, CSV, boundary, profile, and summary files")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp11_convection_diffusion.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

mode = "all" #@param ["all", "forced_plate", "forced_pipe", "natural_plate"]
nx = 121 #@param {type:"integer"}
ny = 61 #@param {type:"integer"}
Pe = 200.0 #@param {type:"number"}
iterations = 3000 #@param {type:"integer"}
output_every = 300 #@param {type:"integer"}

QUICK_ARGS = ['all', 25, 21, 100, 8, 8]
LAB_ARGS = ['all', 121, 61, 200, 3000, 300]
CUSTOM_ARGS = [mode, nx, ny, Pe, iterations, output_every]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'convective_transport_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
summary = read_csv(OUT / "convective_transport_summary.csv")
labels = [str(value) for value in summary["case"]]
plt.figure(figsize=(8, 4))
plt.bar(labels, summary["outlet_or_mean_temperature"])
plt.ylabel("Outlet or mean temperature")
plt.title("Case comparison")
plt.xticks(rotation=20)
plt.grid(True, axis="y")
plt.show()

plt.figure(figsize=(8, 4))
for profile_file in sorted(OUT.glob("*_profiles.csv")):
    profiles = read_csv(profile_file)
    vertical_mask = np.isclose(profiles["x"], profiles["x"][0])
    plt.plot(profiles["temperature"][vertical_mask], profiles["y"][vertical_mask], label=profile_file.stem.replace("_profiles", ""))
plt.xlabel("Temperature")
plt.ylabel("Profile coordinate y")
plt.title("Extracted temperature profiles")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'forced_flat_plate.vtk', 'temperature')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 11)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Run the three modes at $Pe=50,200,1000$. Compare scalar-field shape, boundary-layer thickness, outlet or mean temperature, wall flux, and residual convergence.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.